# Suite SES — Candidates: a filter fed by a frame stream, ranked by a policy

"The transform candidates must come from a filter fed by a frame stream. That's how Store works, no?"
`new Transforms.Session(transforms, frames)` is that filter ([CAP-074: transforms applied step by step](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-074.md)): `transforms`, visitors of
`Transforms.Transform`s, read once, when it is made; `frames`, the frame stream the caller supplies, each frame a
`Visitors.OfPattern`. The filter is each transform's prototype: a frame passes where it is a frame of the prototype.
A passing frame becomes candidates, one per combination of the values of the parameters whose domains are finite, any
other left open, in a canonical order; a follow-up call, `session.ranked(policy)`, ranks them by the policy, whose
clauses name transforms by object reference.
Frames are consumed once, and nothing is queried again: "In a streaming system termination conditions are
relative." The caller takes a candidate by applying it (`Transforms.apply`, then `Transforms.certify`) with its own visitors.

The tests play the caller: each makes its frame stream within its own transactions, from frames it obtained from its
stores (`support.bear`). The fixture is suite XF's shelf, with a note on each item, `shelf.Case` (a union of two
options, `upper` and `lower`) and `shelf.Size` (a union one of whose branches holds a value). Its patterns:
`shelf.Unlabelled`, an item without a label, declaring `case`; `shelf.Unnoted`, an item without a note, declaring
`loud` (a bool) and `note` (a str); `shelf.Unsized`, declaring `size` and `like` (for an object); `shelf.Any`, any
item; and `shelf.Whole`, the shelf.

In [ ]:
import { Builders as B, Parametrics, Proxies, References } from "@mbse/schemas/Framework";
import { repr } from "@mbse/schemas/Framework/Repr";

import { Transforms } from "../src/index.js";

import { assert, bear, equal, raises } from "./support.js";

/** shelf.Shelf, a singleton owning shelf.Items by key (shelf.Holds), each with a name, a label and a note;
 * shelf.Case, a union of two options; and shelf.Size, a union one of whose branches holds an item. */
function shelf(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("ses.shelf/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Shelf: S.OfObject, Item: S.OfObject, Holds: S.OfRelation, Upper: S.OfObject,
      Lower: S.OfObject, Case: S.OfUnion, Size: S.OfUnion })
    .spec([
      (_: any) => _.Shelf.name("shelf.Shelf").ref(true).singleton("shelf.TheShelf").adjacencies({
        items: (t: any) => t.relation([_.Holds]).me("shelf") }),
      (_: any) => _.Item.name("shelf.Item").ref(true).properties({
        name: S.OfStr, label: S.OfStr, note: S.OfStr }).adjacencies([
        (t: any) => t.name("shelf").relation([_.Holds]).me("item").owner(true)]),
      (_: any) => _.Holds.name("shelf.Holds").links([(t: any) => t.name("shelf"), (t: any) => t.name("item")])
        .properties({ key: S.OfStr }).uniques([["shelf", "key"], ["item"]]),
      (_: any) => _.Upper.name("shelf.Upper"),
      (_: any) => _.Lower.name("shelf.Lower"),
      (_: any) => _.Case.name("shelf.Case").branches([
        (b: any) => b.name("upper").schema([_.Upper]), (b: any) => b.name("lower").schema([_.Lower])]),
      (_: any) => _.Size.name("shelf.Size").branches([
        (b: any) => b.name("small").schema([_.Lower]), (b: any) => b.name("custom").schema([_.Item])]),
    ])]))]);
}

/** shelf.Unlabelled, declaring case; shelf.Unnoted, declaring loud and note; shelf.Unsized, declaring size and
 * like; shelf.Any; and shelf.Whole. */
function patterns(txn: any): any {
  const [S, P] = [txn.Schemas, txn.Parametrics];
  return txn.uid("ses.patterns/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Unlabelled: S.OfPattern, Unnoted: S.OfPattern, Unsized: S.OfPattern, Any: S.OfPattern,
      Whole: S.OfPattern })
    .spec([
      (_: any) => _.Unlabelled.name("shelf.Unlabelled").symbols({ i: txn.shelf.Item })
        .spec([(f: any) => f.i.label(null)])
        .parameters([P.OfParameters((ps: any) => ps.parameters({
          case: (d: any) => d.parameter((p: any) => p.for_value([txn.shelf.Case])) }))]),
      (_: any) => _.Unnoted.name("shelf.Unnoted").symbols({ i: txn.shelf.Item })
        .spec([(f: any) => f.i.note(null)])
        .parameters([P.OfParameters((ps: any) => ps.parameters({
          loud: (d: any) => d.parameter((p: any) => p.for_value([S.OfBool])),
          note: (d: any) => d.parameter((p: any) => p.for_value([S.OfStr])) }))]),
      (_: any) => _.Unsized.name("shelf.Unsized").symbols({ i: txn.shelf.Item })
        .parameters([P.OfParameters((ps: any) => ps.parameters({
          size: (d: any) => d.parameter((p: any) => p.for_value([txn.shelf.Size])),
          like: (d: any) => d.parameter((p: any) => p.for_object([txn.shelf.Item])) }))]),
      (_: any) => _.Any.name("shelf.Any").symbols({ i: txn.shelf.Item }),
      (_: any) => _.Whole.name("shelf.Whole").symbols({ s: txn.shelf.Shelf }),
    ])]))]);
}

/** A store of the shelf's schemas and patterns, its shelf holding an item of each name, keyed by it. */
function model(...names: string[]): any {
  const store: any = new B.Store();
  store.transaction(Parametrics.install);
  store.transaction(shelf);
  store.transaction(patterns);
  store.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ s: t.shelf.Shelf })]))]));
  for (const name of names) {
    store.transaction(() => {
      store.shelf.Shelf(store.singleton("shelf.TheShelf"), (s: any) => s.items({
        [name]: (e: any) => e.item((i: any) => i.name(name)) }));
    });
  }
  return store;
}

/** Each item on the shelf: its key, name, label and note. */
function items(store: any): unknown[][] {
  return store.transaction(() => [...store.singleton("shelf.TheShelf").items].map((e: any) =>
    [e.key.get(), e.item.name.get(), e.item.label.get(), e.item.note.get()]));
}

/** The frames of each pattern named, as `store.transaction` gives them back, each with its pattern's name. */
function obtained(store: any, ...names: string[]): [string, any][] {
  return names.flatMap((name) => {
    const held = store.registered(name);
    return [...store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([held]))]))]
      .map((frame: any): [string, any] => [name, frame]);
  });
}

/** The caller's frame stream, within the store's transaction: each frame obtained, as a `Visitors.OfPattern`,
 * made as it is pulled. */
function* stream(store: any, frames: Iterable<[string, any]>): Generator<any> {
  for (const [name, frame] of frames) yield bear(store, name, frame);
}

/** The frames obtained, each pattern's name appended to `pulled` as it is pulled. */
function* counting(frames: Iterable<[string, any]>, pulled: string[]): Generator<[string, any]> {
  for (const [name, frame] of frames) {
    pulled.push(name);
    yield [name, frame];
  }
}

/** Each frame's candidates: each one's transform's name, its frame's pattern and item, its assignments and its
 * open parameters. */
function shown(groups: Transforms.Candidate[][]): unknown[][][] {
  return groups.map((group) => group.map((c) => [c.name, c.frame.as_schema().name, String(c.frame.i.name),
    { ...c.assignments }, c.open]));
}

/** The catalog's transforms, by name, within the store's transaction. */
function catalog(store: any): Record<string, any> {
  return Object.fromEntries([...store.singleton("Transforms.TheCatalog").transforms].map((e: any) => [String(e.name), e.transform]));
}

const transforms: any = model(); // the transforms' store: the schemas and patterns, no data, and the transforms
transforms.transaction(Transforms.install);
for (const define of [
  (t: any) => Transforms.define(t, "Stock", "stock", t.shelf.Any),
  (t: any) => Transforms.define(t, "Label", "label", t.shelf.Unlabelled),
  (t: any) => Transforms.define(t, "Note", "note", t.shelf.Unnoted),
  (t: any) => Transforms.define(t, "Size", "note", t.shelf.Unsized),
  (t: any) => Transforms.define(t, "Copy", "copy", t.shelf.Unlabelled),
]) {
  transforms.transaction(define);
}
const certs: any = new B.Store(); // the certificates' store, as the caller gives it
certs.transaction(Transforms.install);
const procedures = new Transforms.Procedures();

/** Labels the sync's item with its name, in the case its parameter assignment gives. */
function label(source: any, sync: any, target: any, assignments: Transforms.Assignments): void {
  const name = String(sync.i.name);
  Proxies.store_of(sync.i).shelf.Item(sync.i, (i: any) => i.label(assignments.case === "upper" ? name.toUpperCase() : name));
}

procedures.register("label", label);

/** Notes the sync's item, loudly or not. */
function note(source: any, sync: any, target: any, assignments: Transforms.Assignments): void {
  const text = assignments.loud ? (assignments.note as string).toUpperCase() : assignments.note;
  Proxies.store_of(sync.i).shelf.Item(sync.i, (i: any) => i.note(text));
}

procedures.register("note", note);

/** Puts on the target's shelf an item named as the source's, in the case its parameter assignment gives. */
function copy(source: any, sync: any, target: any, assignments: Transforms.Assignments): void {
  let name = String(source.i.name);
  name = assignments.case === "upper" ? name.toUpperCase() : name;
  Proxies.store_of(target.s).shelf.Shelf(target.s, (s: any) => s.items({ [name]: (e: any) => e.item((i: any) => i.name(name)) }));
}

procedures.register("copy", copy);

<a id="SES-01"></a>
## SES-01 · The filter is each transform's prototype: a frame passes where it is a frame of the prototype, and becomes candidates, one per combination of its finite parameters' values (a bool's; a union of options' names), any other parameter left open, in the canonical order

In [ ]:
{
  const data = model("bolt", "nut");
  const frames = obtained(data, "shelf.Unlabelled", "shelf.Whole", "shelf.Unnoted", "shelf.Unsized", "shelf.Any");
  const [groups, view] = transforms.transaction(() => data.transaction(() => {
    const named = catalog(transforms);
    const session = new Transforms.Session(["Stock", "Label", "Note", "Size"].map((n) => named[n]), stream(data, frames));
    const groups = [...session];
    return [groups, shown(groups)]; // read within the caller's transaction, where its visitors live
  }));
  assert(equal(view, [ // a frame's candidates: the transforms' order, then each value's in its domain
    [["Label", "shelf.Unlabelled", "bolt", { case: "upper" }, []], ["Label", "shelf.Unlabelled", "bolt", { case: "lower" }, []]],
    [["Label", "shelf.Unlabelled", "nut", { case: "upper" }, []], ["Label", "shelf.Unlabelled", "nut", { case: "lower" }, []]],
    // shelf.Whole's frame passes no filter
    [["Note", "shelf.Unnoted", "bolt", { loud: false }, ["note"]], ["Note", "shelf.Unnoted", "bolt", { loud: true }, ["note"]]],
    [["Note", "shelf.Unnoted", "nut", { loud: false }, ["note"]], ["Note", "shelf.Unnoted", "nut", { loud: true }, ["note"]]],
    [["Size", "shelf.Unsized", "bolt", {}, ["size", "like"]]], [["Size", "shelf.Unsized", "nut", {}, ["size", "like"]]], // neither finite
    [["Stock", "shelf.Any", "bolt", {}, []]], [["Stock", "shelf.Any", "nut", {}, []]], // no parameters
  ]));
  const first = groups[0][0];
  assert(first.score === 0.0 && repr(first).startsWith("Candidate(transform=") && !repr(first).includes("frame"));
  const loud = groups[2][1];
  const answered = loud.answer({ note: "fragile" }); // an open parameter assigned
  assert(equal([{ ...answered.assignments }, answered.open], [{ loud: true, note: "fragile" }, []]));
  assert(equal(loud.open, ["note"])); // a candidate is a value: answering gives another
  raises(TypeError, "transform 'Note' has no open parameter 'loud'", () => loud.answer({ loud: false }));
}

<a id="SES-02"></a>
## SES-02 · A session consumes its stream: each frame is pulled once, as the caller iterates, and nothing is queried again; it ends when the stream does

In [ ]:
{
  const data = model("bolt", "nut");
  const frames = obtained(data, "shelf.Unlabelled");
  const pulled: string[] = [];
  transforms.transaction(() => data.transaction(() => {
    const session = new Transforms.Session([catalog(transforms).Label], stream(data, counting(frames, pulled)));
    assert(equal(pulled, [])); // nothing pulled until the caller iterates
    const each = session[Symbol.iterator]();
    assert(equal((each.next().value as Transforms.Candidate[]).map((c) => String(c.frame.i.name)), ["bolt", "bolt"])
      && pulled.length === 1); // one frame at a time
    assert(equal((each.next().value as Transforms.Candidate[]).map((c) => String(c.frame.i.name)), ["nut", "nut"])
      && pulled.length === 2);
    assert(equal([...each], []) && equal([...session], [])); // the stream drained: termination, relative to it
    assert(equal([...session.ranked(new Transforms.Policy())], [])); // iterated once, directly or ranked
  }));
  assert(pulled.length === 2 && equal(items(data), [["bolt", "bolt", null, null], ["nut", "nut", null, null]]));
}

<a id="SES-03"></a>
## SES-03 · A policy ranks each frame's candidates: a clause weighs the candidates of the transforms it names by object reference whose assignments hold its own; a score is the sum of the weights, ties in the canonical order

In [ ]:
{
  const data = model("bolt");
  const frames = obtained(data, "shelf.Unlabelled", "shelf.Unnoted");
  const ranked: Transforms.Candidate[][] = transforms.transaction(() => data.transaction(() => {
    const named = catalog(transforms);
    const [Label, Copy, Note] = ["Label", "Copy", "Note"].map((n) => References.of(transforms, named[n]));
    const policy = new Transforms.Policy(
      new Transforms.Clause(Label, { assignments: { case: "lower" }, weight: 2.0 }),
      new Transforms.Clause([Label, Copy], { weight: 0.5 }), // one clause, several transforms: those sharing a reason
      new Transforms.Clause(Note, { assignments: { note: "fragile" } }), // note is open: no candidate's assignments hold it
      new Transforms.Clause(Copy, { assignments: { size: "small" } }), // Copy has no size: it weighs none of its candidates
    );
    return [...new Transforms.Session(["Copy", "Label", "Note"].map((n) => named[n]), stream(data, frames)).ranked(policy)];
  }));
  assert(equal(ranked.map((group) => group.map((c) => [c.name, { ...c.assignments }, c.score])), [
    [["Label", { case: "lower" }, 2.5], // ranked first; then ties, in the canonical order
      ["Copy", { case: "upper" }, 0.5], ["Copy", { case: "lower" }, 0.5], ["Label", { case: "upper" }, 0.5]],
    [["Note", { loud: false }, 0.0], ["Note", { loud: true }, 0.0]],
  ]));
  const elsewhere = model(); // a reference is the catalog's: another store's transform named Label is another transform
  elsewhere.transaction(Transforms.install);
  elsewhere.transaction((t: any) => Transforms.define(t, "Label", "label", t.shelf.Unlabelled));
  const other = elsewhere.transaction(() => References.of(elsewhere, catalog(elsewhere).Label));
  assert(new Transforms.Policy(new Transforms.Clause(other)).score(ranked[0]![0]!) === 0);
  raises(TypeError, "a clause names a transform by its reference, References.of(store, transform), or a sequence of them",
    () => new Transforms.Clause("Label"));
}

<a id="SES-04"></a>
## SES-04 · The caller takes a candidate by applying it with its own visitors: the candidate's frame as the argument the caller chooses, its open parameters answered, and who decided it recorded

In [ ]:
{
  const [source, sync, target] = [model("bolt", "nut"), model("bolt", "nut"), model()]; // the caller's copies
  const [in_sync, in_source] = [obtained(sync, "shelf.Unnoted"), obtained(source, "shelf.Unlabelled")];
  const [[, shelf_frame]] = obtained(target, "shelf.Whole");
  transforms.transaction(() => sync.transaction(() => {
    const named = catalog(transforms);
    const policy = new Transforms.Policy(new Transforms.Clause(References.of(transforms, named.Note), { assignments: { loud: true } }));
    for (const group of new Transforms.Session([named.Note], stream(sync, in_sync)).ranked(policy)) {
      const taken = group[0]!.answer({ note: "fragile" }); // the policy's first, its open parameter answered by the caller
      Transforms.apply(named.Note, taken.assignments, procedures, { sync: taken.frame });
      Transforms.certify(certs, named.Note, taken.assignments, "taken", "policy", { sync: taken.frame });
    }
  }));
  assert(equal(items(sync), [["bolt", "bolt", null, "FRAGILE"], ["nut", "nut", null, "FRAGILE"]]) && items(source)[0]![3] === null);
  transforms.transaction(() => source.transaction(() => target.transaction(() => {
    const named = catalog(transforms);
    const whole = bear(target, "shelf.Whole", shelf_frame);
    for (const group of new Transforms.Session([named.Copy], stream(source, in_source))) {
      if (String(group[0]!.frame.i.name) === "nut") { // the caller decides: nut, upper; bolt left untaken
        Transforms.apply(named.Copy, group[0]!.assignments, procedures, { source: group[0]!.frame, target: whole });
        Transforms.certify(certs, named.Copy, group[0]!.assignments, "taken", "caller", { source: group[0]!.frame,
          target: whole });
      }
    }
  })));
  assert(equal(items(target), [["NUT", "NUT", null, null]]));
  certs.transaction(() => {
    const history = certs.singleton("Transforms.TheHistories").histories.at("taken").history;
    assert(equal([...history.decisions].map((d: any) => String(d.by)), ["policy", "policy", "caller"]));
    assert(equal([...history.steps].map((s: any) => [...s.step.frames].map((e: any) => String(e.argument))),
      [["sync"], ["sync"], ["source", "target"]]));
  });
}